# Beans Shield — Treinamento do Modelo de Detecção de Fraude

**Projeto:** beans-shield (Go library)  
**Ambiente:** Google Colab Enterprise Plus  
**Objetivo:** Treinar um modelo LightGBM para detecção de fraude em transações Pix/betting e exportá-lo em formato ONNX para inferência no backend Go.

---

## Visão Geral do Pipeline

1. Geração de dados sintéticos realistas (100k transações)
2. Feature engineering e análise exploratória
3. Treinamento com LightGBM (cross-validation + early stopping)
4. Avaliação com métricas operacionais
5. Exportação ONNX para integração com Go
6. Exportação alternativa JSON para ThresholdScorer

---
## 1. Setup & Imports

In [ ]:
# Install dependencies
!pip install -q lightgbm onnx onnxmltools skl2onnx pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from pathlib import Path

import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    roc_curve, precision_recall_curve, confusion_matrix,
    classification_report, average_precision_score
)
import onnxmltools
from onnxmltools.convert import convert_lightgbm
from onnxconverter_common import FloatTensorType
import onnx

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Plot style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

print(f"LightGBM version: {lgb.__version__}")
print(f"ONNX version: {onnx.__version__}")

---
## 2. Configuração do Dataset

Geramos **100.000 transações sintéticas** que simulam padrões reais de transações Pix e apostas (betting) observados na Beans Capital.

### Lógica de rotulação (is_fraud):
- **Alto valor + destino novo + horário incomum** → 80% de probabilidade de fraude
- **Alta velocidade + alto valor** → 70% de probabilidade de fraude  
- **Padrões normais** → 2% de probabilidade de fraude (taxa base realista)

### Features (devem corresponder exatamente ao código Go):
```
amount_normalized, hour_sin, hour_cos, is_pix_out, is_pix_in,
is_withdrawal, is_swap, is_crypto_buy, velocity_count_1h,
velocity_count_24h, velocity_amount_24h, is_new_destination,
is_unusual_hour, amount_vs_avg_24h
```

In [ ]:
# Feature names — MUST match the Go code exactly
FEATURE_NAMES = [
    "amount_normalized",
    "hour_sin",
    "hour_cos",
    "is_pix_out",
    "is_pix_in",
    "is_withdrawal",
    "is_swap",
    "is_crypto_buy",
    "velocity_count_1h",
    "velocity_count_24h",
    "velocity_amount_24h",
    "is_new_destination",
    "is_unusual_hour",
    "amount_vs_avg_24h",
]

N_SAMPLES = 100_000


def generate_synthetic_transactions(n: int) -> pd.DataFrame:
    """Generate synthetic Pix/betting transactions with realistic fraud patterns."""

    # --- Amount (log-normal distribution) ---
    # Most transactions are small (R$10-500), some are large (R$5k-50k)
    raw_amounts = np.random.lognormal(mean=5.0, sigma=1.5, size=n)
    raw_amounts = np.clip(raw_amounts, 1, 200_000)
    amount_normalized = np.log1p(raw_amounts) / np.log1p(200_000)  # Normalize to [0, 1]

    # --- Hour (cyclical encoding) ---
    hours = np.random.choice(24, size=n, p=[
        0.01, 0.01, 0.01, 0.01, 0.01, 0.02,  # 0-5h (low activity)
        0.03, 0.05, 0.07, 0.08, 0.08, 0.07,  # 6-11h (morning)
        0.06, 0.06, 0.06, 0.06, 0.06, 0.05,  # 12-17h (afternoon)
        0.05, 0.05, 0.04, 0.03, 0.02, 0.01,  # 18-23h (evening)
    ])
    hour_sin = np.sin(2 * np.pi * hours / 24)
    hour_cos = np.cos(2 * np.pi * hours / 24)

    # --- Transaction type (one-hot) ---
    tx_types = np.random.choice(
        ["pix_out", "pix_in", "withdrawal", "swap", "crypto_buy"],
        size=n,
        p=[0.40, 0.30, 0.10, 0.10, 0.10]
    )
    is_pix_out = (tx_types == "pix_out").astype(float)
    is_pix_in = (tx_types == "pix_in").astype(float)
    is_withdrawal = (tx_types == "withdrawal").astype(float)
    is_swap = (tx_types == "swap").astype(float)
    is_crypto_buy = (tx_types == "crypto_buy").astype(float)

    # --- Velocity features ---
    velocity_count_1h = np.random.exponential(scale=2.0, size=n).astype(int)
    velocity_count_1h = np.clip(velocity_count_1h, 0, 50)

    velocity_count_24h = velocity_count_1h * np.random.uniform(3, 10, size=n)
    velocity_count_24h = velocity_count_24h.astype(int)
    velocity_count_24h = np.clip(velocity_count_24h, 0, 200)

    velocity_amount_24h = raw_amounts * np.random.uniform(1, 8, size=n)
    velocity_amount_24h = np.log1p(velocity_amount_24h) / np.log1p(1_000_000)  # Normalize

    # --- Binary features ---
    is_new_destination = np.random.binomial(1, 0.15, size=n).astype(float)
    is_unusual_hour = ((hours >= 0) & (hours <= 5)).astype(float)

    # --- Ratio feature ---
    amount_vs_avg_24h = np.random.lognormal(mean=0, sigma=0.8, size=n)
    amount_vs_avg_24h = np.clip(amount_vs_avg_24h, 0.01, 50.0)

    # --- Fraud labeling with realistic patterns ---
    fraud_prob = np.full(n, 0.02)  # Base rate: 2%

    # Pattern 1: High amount + new destination + unusual hour → 80% fraud
    high_amount = amount_normalized > 0.7
    pattern_1 = high_amount & (is_new_destination == 1) & (is_unusual_hour == 1)
    fraud_prob[pattern_1] = 0.80

    # Pattern 2: High velocity + high amount → 70% fraud
    high_velocity = velocity_count_1h >= 5
    pattern_2 = high_velocity & high_amount & ~pattern_1
    fraud_prob[pattern_2] = 0.70

    # Pattern 3: Very high amount_vs_avg + new destination → 60% fraud
    pattern_3 = (amount_vs_avg_24h > 5.0) & (is_new_destination == 1) & ~pattern_1 & ~pattern_2
    fraud_prob[pattern_3] = 0.60

    # Pattern 4: Pix out + unusual hour + new destination → 50% fraud
    pattern_4 = (is_pix_out == 1) & (is_unusual_hour == 1) & (is_new_destination == 1) & ~pattern_1 & ~pattern_2 & ~pattern_3
    fraud_prob[pattern_4] = 0.50

    # Pattern 5: High velocity alone → 15% fraud
    pattern_5 = high_velocity & ~high_amount & ~pattern_1 & ~pattern_2 & ~pattern_3 & ~pattern_4
    fraud_prob[pattern_5] = 0.15

    is_fraud = np.random.binomial(1, fraud_prob).astype(float)

    # Build DataFrame
    df = pd.DataFrame({
        "amount_normalized": amount_normalized,
        "hour_sin": hour_sin,
        "hour_cos": hour_cos,
        "is_pix_out": is_pix_out,
        "is_pix_in": is_pix_in,
        "is_withdrawal": is_withdrawal,
        "is_swap": is_swap,
        "is_crypto_buy": is_crypto_buy,
        "velocity_count_1h": velocity_count_1h.astype(float),
        "velocity_count_24h": velocity_count_24h.astype(float),
        "velocity_amount_24h": velocity_amount_24h,
        "is_new_destination": is_new_destination,
        "is_unusual_hour": is_unusual_hour,
        "amount_vs_avg_24h": amount_vs_avg_24h,
        "is_fraud": is_fraud,
    })

    return df


# Generate dataset
df = generate_synthetic_transactions(N_SAMPLES)

print(f"Dataset shape: {df.shape}")
print(f"\nDistribuição de fraude:")
print(df["is_fraud"].value_counts())
print(f"\nTaxa de fraude: {df['is_fraud'].mean():.2%}")
print(f"\nEstatísticas descritivas:")
df.describe().round(3)

---
## 3. Feature Engineering & EDA

Análise exploratória para validar que os dados sintéticos apresentam padrões realistas e que as features são discriminativas.

In [ ]:
# --- Feature distributions ---
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle("Distribuição das Features Numéricas", fontsize=14, fontweight="bold")

numeric_features = [
    "amount_normalized", "velocity_count_1h", "velocity_count_24h",
    "velocity_amount_24h", "amount_vs_avg_24h", "hour_sin",
    "hour_cos", "is_new_destination", "is_unusual_hour"
]

for idx, feat in enumerate(numeric_features):
    ax = axes[idx // 3, idx % 3]
    df[df["is_fraud"] == 0][feat].hist(ax=ax, bins=50, alpha=0.6, label="Legítima", color="steelblue")
    df[df["is_fraud"] == 1][feat].hist(ax=ax, bins=50, alpha=0.6, label="Fraude", color="crimson")
    ax.set_title(feat)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# --- Fraud rate by feature ---
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("Taxa de Fraude por Feature", fontsize=14, fontweight="bold")

# Amount bins
ax = axes[0, 0]
df["amount_bin"] = pd.qcut(df["amount_normalized"], q=10, duplicates="drop")
df.groupby("amount_bin")["is_fraud"].mean().plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Fraude por Faixa de Valor")
ax.set_ylabel("Taxa de Fraude")
ax.tick_params(axis="x", rotation=45)

# Velocity 1h
ax = axes[0, 1]
df["vel_1h_bin"] = pd.cut(df["velocity_count_1h"], bins=[0, 1, 3, 5, 10, 50])
df.groupby("vel_1h_bin")["is_fraud"].mean().plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Fraude por Velocidade (1h)")
ax.set_ylabel("Taxa de Fraude")

# New destination
ax = axes[0, 2]
df.groupby("is_new_destination")["is_fraud"].mean().plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Fraude por Destino Novo")
ax.set_ylabel("Taxa de Fraude")
ax.set_xticklabels(["Conhecido", "Novo"], rotation=0)

# Unusual hour
ax = axes[1, 0]
df.groupby("is_unusual_hour")["is_fraud"].mean().plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Fraude por Horário Incomum")
ax.set_ylabel("Taxa de Fraude")
ax.set_xticklabels(["Normal", "Incomum (0-5h)"], rotation=0)

# Transaction type
ax = axes[1, 1]
tx_cols = ["is_pix_out", "is_pix_in", "is_withdrawal", "is_swap", "is_crypto_buy"]
fraud_by_type = {}
for col in tx_cols:
    fraud_by_type[col.replace("is_", "")] = df[df[col] == 1]["is_fraud"].mean()
pd.Series(fraud_by_type).plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Fraude por Tipo de Transação")
ax.set_ylabel("Taxa de Fraude")
ax.tick_params(axis="x", rotation=45)

# Amount vs avg
ax = axes[1, 2]
df["avg_ratio_bin"] = pd.cut(df["amount_vs_avg_24h"], bins=[0, 1, 2, 5, 10, 50])
df.groupby("avg_ratio_bin")["is_fraud"].mean().plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Fraude por Ratio vs Média 24h")
ax.set_ylabel("Taxa de Fraude")

plt.tight_layout()
plt.show()

# Cleanup temp columns
df.drop(columns=["amount_bin", "vel_1h_bin", "avg_ratio_bin"], inplace=True)

In [ ]:
# --- Correlation matrix ---
fig, ax = plt.subplots(figsize=(12, 10))
corr = df[FEATURE_NAMES + ["is_fraud"]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
    center=0, ax=ax, square=True, linewidths=0.5
)
ax.set_title("Matriz de Correlação (Features + Target)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 4. Treinamento do Modelo

Utilizamos **LightGBM** por ser:
- Extremamente rápido para inferência (crítico para detecção em tempo real)
- Eficiente com dados tabulares
- Facilmente exportável para ONNX
- Robusto a features com diferentes escalas

### Estratégia:
- `scale_pos_weight` para compensar desbalanceamento
- 5-fold stratified cross-validation
- Early stopping no validation set

In [ ]:
# Split features and target
X = df[FEATURE_NAMES].values
y = df["is_fraud"].values

# Train/test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Calculate scale_pos_weight
n_negative = (y_train == 0).sum()
n_positive = (y_train == 1).sum()
scale_pos_weight = n_negative / n_positive

print(f"Train set: {len(X_train)} samples ({y_train.sum():.0f} fraudes, {y_train.mean():.2%})")
print(f"Test set:  {len(X_test)} samples ({y_test.sum():.0f} fraudes, {y_test.mean():.2%})")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

In [ ]:
# LightGBM parameters
params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "num_leaves": 63,
    "max_depth": 8,
    "learning_rate": 0.05,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "scale_pos_weight": scale_pos_weight,
    "min_child_samples": 50,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "verbose": -1,
    "seed": SEED,
    "n_jobs": -1,
}

# --- 5-Fold Cross-Validation ---
print("=" * 60)
print("5-Fold Stratified Cross-Validation")
print("=" * 60)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = []
cv_models = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]

    dtrain = lgb.Dataset(X_fold_train, label=y_fold_train, feature_name=FEATURE_NAMES)
    dval = lgb.Dataset(X_fold_val, label=y_fold_val, feature_name=FEATURE_NAMES, reference=dtrain)

    model = lgb.train(
        params,
        dtrain,
        num_boost_round=1000,
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=0),  # Suppress per-round output
        ],
    )

    val_pred = model.predict(X_fold_val)
    auc = roc_auc_score(y_fold_val, val_pred)
    cv_scores.append(auc)
    cv_models.append(model)

    print(f"  Fold {fold}: AUC = {auc:.4f} | Best iteration: {model.best_iteration}")

print(f"\n  CV Mean AUC: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})")

In [ ]:
# --- Train final model on full training set ---
print("\n" + "=" * 60)
print("Treinamento do Modelo Final")
print("=" * 60)

# Use a small validation split for early stopping
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=SEED, stratify=y_train
)

dtrain_final = lgb.Dataset(X_tr, label=y_tr, feature_name=FEATURE_NAMES)
dval_final = lgb.Dataset(X_val, label=y_val, feature_name=FEATURE_NAMES, reference=dtrain_final)

final_model = lgb.train(
    params,
    dtrain_final,
    num_boost_round=1000,
    valid_sets=[dval_final],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100),
    ],
)

print(f"\nMelhor iteração: {final_model.best_iteration}")

# --- Evaluate on test set ---
y_pred_proba = final_model.predict(X_test)
y_pred = (y_pred_proba >= 0.5).astype(int)

print(f"\n{'=' * 60}")
print(f"Métricas no Test Set (threshold = 0.5)")
print(f"{'=' * 60}")
print(f"  AUC-ROC:   {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"  AUC-PR:    {average_precision_score(y_test, y_pred_proba):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"  F1-Score:  {f1_score(y_test, y_pred):.4f}")

---
## 5. Avaliação

Avaliação completa com curvas ROC, Precision-Recall, importância de features e métricas operacionais.

In [ ]:
# --- ROC Curve ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
fpr, tpr, roc_thresholds = roc_curve(y_test, y_pred_proba)
auc_score = roc_auc_score(y_test, y_pred_proba)

ax = axes[0]
ax.plot(fpr, tpr, color="steelblue", lw=2, label=f"AUC = {auc_score:.4f}")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random")
ax.set_xlabel("Taxa de Falsos Positivos (FPR)")
ax.set_ylabel("Taxa de Verdadeiros Positivos (TPR)")
ax.set_title("Curva ROC", fontweight="bold")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)

# Precision-Recall
precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_test, y_pred_proba)
ap_score = average_precision_score(y_test, y_pred_proba)

ax = axes[1]
ax.plot(recall_curve, precision_curve, color="crimson", lw=2, label=f"AP = {ap_score:.4f}")
ax.axhline(y=y_test.mean(), color="k", linestyle="--", lw=1, label=f"Baseline = {y_test.mean():.3f}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Curva Precision-Recall", fontweight="bold")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- Feature Importance ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gain importance
importance_gain = final_model.feature_importance(importance_type="gain")
importance_df = pd.DataFrame({
    "feature": FEATURE_NAMES,
    "importance": importance_gain
}).sort_values("importance", ascending=True)

ax = axes[0]
ax.barh(importance_df["feature"], importance_df["importance"], color="steelblue")
ax.set_title("Feature Importance (Gain)", fontweight="bold")
ax.set_xlabel("Gain")

# Split importance
importance_split = final_model.feature_importance(importance_type="split")
importance_df2 = pd.DataFrame({
    "feature": FEATURE_NAMES,
    "importance": importance_split
}).sort_values("importance", ascending=True)

ax = axes[1]
ax.barh(importance_df2["feature"], importance_df2["importance"], color="darkorange")
ax.set_title("Feature Importance (Split)", fontweight="bold")
ax.set_xlabel("Number of Splits")

plt.tight_layout()
plt.show()

In [ ]:
# --- Optimal threshold (maximize F1) ---
thresholds = np.arange(0.1, 0.9, 0.01)
f1_scores = []

for t in thresholds:
    y_pred_t = (y_pred_proba >= t).astype(int)
    f1_scores.append(f1_score(y_test, y_pred_t))

optimal_threshold = thresholds[np.argmax(f1_scores)]
best_f1 = max(f1_scores)

print(f"Threshold ótimo (max F1): {optimal_threshold:.2f}")
print(f"F1-Score no threshold ótimo: {best_f1:.4f}")

# Confusion matrix at optimal threshold
y_pred_optimal = (y_pred_proba >= optimal_threshold).astype(int)
cm = confusion_matrix(y_test, y_pred_optimal)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", ax=ax,
    xticklabels=["Legítima", "Fraude"],
    yticklabels=["Legítima", "Fraude"]
)
ax.set_xlabel("Predição")
ax.set_ylabel("Real")
ax.set_title(f"Matriz de Confusão (threshold = {optimal_threshold:.2f})", fontweight="bold")
plt.tight_layout()
plt.show()

# --- Operational Metrics ---
tn, fp, fn, tp = cm.ravel()
fraud_blocked = tp / (tp + fn)  # Recall
false_positive_rate = fp / (fp + tn)

print(f"\n{'=' * 60}")
print(f"MÉTRICAS OPERACIONAIS (threshold = {optimal_threshold:.2f})")
print(f"{'=' * 60}")
print(f"  Fraudes bloqueadas:    {fraud_blocked:.1%} ({tp} de {tp+fn})")
print(f"  Taxa de falso positivo: {false_positive_rate:.2%} ({fp} de {fp+tn})")
print(f"  Transações bloqueadas indevidamente: {fp}")
print(f"  Fraudes não detectadas: {fn}")
print(f"{'=' * 60}")

---
## 6. Exportação ONNX

Exportamos o modelo treinado para o formato **ONNX** (Open Neural Network Exchange), que pode ser carregado diretamente pelo runtime Go via `onnxruntime-go`.

### Vantagens do ONNX:
- Inferência ultra-rápida (< 1ms por transação)
- Runtime disponível para Go, Python, C++, etc.
- Modelo self-contained (não depende do LightGBM em produção)

In [ ]:
# --- Convert to ONNX ---
OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(exist_ok=True)

ONNX_PATH = OUTPUT_DIR / "beans_shield_model.onnx"

# Define input type: float32 tensor with 14 features
initial_type = [("features", FloatTensorType([None, len(FEATURE_NAMES)]))]

# Convert LightGBM model to ONNX
onnx_model = convert_lightgbm(
    final_model,
    initial_types=initial_type,
    target_opset=15,
)

# Add metadata
meta = onnx_model.metadata_props.add()
meta.key = "model_name"
meta.value = "beans_shield_fraud_detection"

meta = onnx_model.metadata_props.add()
meta.key = "version"
meta.value = "1.0.0"

meta = onnx_model.metadata_props.add()
meta.key = "feature_names"
meta.value = json.dumps(FEATURE_NAMES)

meta = onnx_model.metadata_props.add()
meta.key = "optimal_threshold"
meta.value = str(optimal_threshold)

# Validate and save
onnx.checker.check_model(onnx_model)
onnx.save(onnx_model, str(ONNX_PATH))

# Print model info
model_size = os.path.getsize(ONNX_PATH)
print(f"Modelo ONNX salvo em: {ONNX_PATH}")
print(f"Tamanho do modelo: {model_size / 1024:.1f} KB ({model_size / (1024*1024):.2f} MB)")
print(f"Threshold recomendado: {optimal_threshold:.2f}")
print(f"Features: {len(FEATURE_NAMES)}")

In [ ]:
# --- Inference test with ONNX ---
import onnxruntime as ort
import time

# Load ONNX model
session = ort.InferenceSession(str(ONNX_PATH))

# Get input/output names
input_name = session.get_inputs()[0].name
output_names = [o.name for o in session.get_outputs()]

print(f"Input name: {input_name}")
print(f"Output names: {output_names}")

# Test inference
test_input = X_test[:1].astype(np.float32)
result = session.run(None, {input_name: test_input})
print(f"\nTeste de inferência:")
print(f"  Input shape: {test_input.shape}")
print(f"  Output (label): {result[0]}")
print(f"  Output (probabilities): {result[1]}")

# Benchmark latency
n_iterations = 1000
single_input = X_test[:1].astype(np.float32)

start = time.perf_counter()
for _ in range(n_iterations):
    session.run(None, {input_name: single_input})
elapsed = time.perf_counter() - start

print(f"\nBenchmark de latência ({n_iterations} iterações):")
print(f"  Latência média: {elapsed / n_iterations * 1000:.3f} ms")
print(f"  Throughput: {n_iterations / elapsed:.0f} inferências/segundo")

# Batch benchmark
batch_input = X_test[:100].astype(np.float32)
start = time.perf_counter()
for _ in range(100):
    session.run(None, {input_name: batch_input})
elapsed_batch = time.perf_counter() - start

print(f"\nBatch (100 transações):")
print(f"  Latência média: {elapsed_batch / 100 * 1000:.3f} ms")
print(f"  Throughput: {100 * 100 / elapsed_batch:.0f} transações/segundo")

In [ ]:
# --- Verify ONNX output matches LightGBM ---
# This ensures the conversion is lossless
onnx_preds = []
batch_size = 1000

for i in range(0, len(X_test), batch_size):
    batch = X_test[i:i+batch_size].astype(np.float32)
    result = session.run(None, {input_name: batch})
    # result[1] contains probabilities as list of dicts [{0: prob_0, 1: prob_1}, ...]
    probs = [r[1] for r in result[1]]
    onnx_preds.extend(probs)

onnx_preds = np.array(onnx_preds)
max_diff = np.max(np.abs(onnx_preds - y_pred_proba))
mean_diff = np.mean(np.abs(onnx_preds - y_pred_proba))

print(f"Validação ONNX vs LightGBM:")
print(f"  Diferença máxima: {max_diff:.8f}")
print(f"  Diferença média:  {mean_diff:.8f}")
print(f"  Status: {'OK - Conversão lossless' if max_diff < 1e-5 else 'ATENÇÃO - Diferenças significativas'}")

print(f"\n" + "=" * 60)
print(f"Para copiar o modelo para o projeto Go:")
print(f"  cp {ONNX_PATH} /path/to/beans-shield/model/beans_shield_model.onnx")
print(f"=" * 60)

---
## 7. Exportação Alternativa (JSON weights)

Para deploys mais simples que não podem usar o ONNX runtime, exportamos os pesos do modelo em formato JSON que pode ser carregado pelo `ThresholdScorer` no Go.

Este formato é uma simplificação — usa thresholds e pesos lineares derivados da importância das features, servindo como um modelo "lite" para ambientes com restrições.

In [ ]:
# --- Export JSON weights for ThresholdScorer ---

# Normalize feature importance to use as weights
importance = final_model.feature_importance(importance_type="gain")
weights = importance / importance.sum()

# Calculate optimal thresholds per feature using the training data
# For each feature, find the threshold that best separates fraud from legitimate
feature_thresholds = []

df_train = pd.DataFrame(X_train, columns=FEATURE_NAMES)
df_train["is_fraud"] = y_train

for feat in FEATURE_NAMES:
    fraud_mean = df_train[df_train["is_fraud"] == 1][feat].mean()
    legit_mean = df_train[df_train["is_fraud"] == 0][feat].mean()
    # Threshold at midpoint between means (simple but effective)
    threshold = (fraud_mean + legit_mean) / 2
    # Direction: positive if fraud has higher values
    direction = 1.0 if fraud_mean > legit_mean else -1.0
    feature_thresholds.append({
        "feature": feat,
        "threshold": float(round(threshold, 6)),
        "weight": float(round(weights[FEATURE_NAMES.index(feat)], 6)),
        "direction": direction,
        "fraud_mean": float(round(fraud_mean, 4)),
        "legit_mean": float(round(legit_mean, 4)),
    })

# Build export object
json_export = {
    "model_name": "beans_shield_threshold_scorer",
    "version": "1.0.0",
    "description": "Simplified linear scorer derived from LightGBM feature importance",
    "optimal_threshold": float(optimal_threshold),
    "feature_names": FEATURE_NAMES,
    "n_features": len(FEATURE_NAMES),
    "features": feature_thresholds,
    "training_metadata": {
        "n_samples": int(len(X_train)),
        "n_fraud": int(y_train.sum()),
        "fraud_rate": float(round(y_train.mean(), 4)),
        "cv_auc_mean": float(round(np.mean(cv_scores), 4)),
        "cv_auc_std": float(round(np.std(cv_scores), 4)),
        "test_auc": float(round(roc_auc_score(y_test, y_pred_proba), 4)),
    }
}

JSON_PATH = OUTPUT_DIR / "beans_shield_weights.json"
with open(JSON_PATH, "w") as f:
    json.dump(json_export, f, indent=2)

print(f"Modelo JSON salvo em: {JSON_PATH}")
print(f"Tamanho: {os.path.getsize(JSON_PATH) / 1024:.1f} KB")
print(f"\nTop 5 features por peso:")
for ft in sorted(feature_thresholds, key=lambda x: x["weight"], reverse=True)[:5]:
    direction_str = "higher=fraud" if ft["direction"] > 0 else "lower=fraud"
    print(f"  {ft['feature']:25s} peso={ft['weight']:.4f}  threshold={ft['threshold']:.4f}  ({direction_str})")

print(f"\nPara usar no Go ThresholdScorer:")
print(f"  cp {JSON_PATH} /path/to/beans-shield/model/beans_shield_weights.json")

In [ ]:
# --- Validate JSON scorer accuracy ---
# Simulate what the Go ThresholdScorer would do

def json_scorer_predict(X: np.ndarray, features_config: list) -> np.ndarray:
    """Simulate the Go ThresholdScorer inference."""
    scores = np.zeros(len(X))

    for i, feat_config in enumerate(features_config):
        weight = feat_config["weight"]
        threshold = feat_config["threshold"]
        direction = feat_config["direction"]

        feature_values = X[:, i]

        if direction > 0:
            # Higher values = more suspicious
            contribution = np.where(
                feature_values > threshold,
                (feature_values - threshold) * weight * direction,
                0
            )
        else:
            # Lower values = more suspicious
            contribution = np.where(
                feature_values < threshold,
                (threshold - feature_values) * weight * abs(direction),
                0
            )

        scores += contribution

    # Normalize to [0, 1] using sigmoid
    scores = 1 / (1 + np.exp(-scores * 5))  # Scale factor for better separation
    return scores


json_preds = json_scorer_predict(X_test, feature_thresholds)
json_auc = roc_auc_score(y_test, json_preds)

print(f"Comparação de performance:")
print(f"  LightGBM (ONNX): AUC = {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"  JSON Scorer:     AUC = {json_auc:.4f}")
print(f"  Diferença:       {roc_auc_score(y_test, y_pred_proba) - json_auc:.4f}")
print(f"\nNota: O JSON scorer é uma aproximação. Use ONNX para máxima acurácia.")

---
## 8. Próximos Passos

### Como treinar com dados reais da Beans Capital

1. **Coleta de dados:**
   - Exportar transações do banco de dados com labels de fraude confirmada
   - Mínimo recomendado: 50k transações com pelo menos 500 fraudes confirmadas
   - Incluir chargebacks, disputas e bloqueios manuais como labels positivos

2. **Substituir a geração sintética:**
   ```python
   # Carregar dados reais do BigQuery/PostgreSQL
   df = pd.read_sql(query, connection)
   # Ou de um arquivo
   df = pd.read_parquet("gs://beans-data/transactions.parquet")
   ```

3. **Feature engineering adicional:**
   - Device fingerprint features
   - Geolocation anomaly score
   - Historical user behavior embeddings
   - Merchant risk score

---

### Como retreinar periodicamente

1. **Frequência recomendada:** Semanal ou quando a taxa de fraude mudar > 20%
2. **Pipeline sugerido:**
   - Cloud Scheduler → Cloud Function → Vertex AI Training → Model Registry
   - Ou: GitHub Actions com schedule trigger
3. **Monitoramento:**
   - Acompanhar AUC em produção via métricas do modelo
   - Alertar se AUC cair abaixo de 0.90
   - Detectar data drift com Evidently ou similar

---

### Como fazer A/B test (modelo vs regras)

1. **Shadow mode:**
   - Rodar modelo em paralelo com regras existentes
   - Logar predições do modelo sem bloquear transações
   - Comparar: modelo detectaria fraudes que as regras perderam?

2. **Gradual rollout:**
   - 10% do tráfego → modelo (com fallback para regras)
   - Monitorar falsos positivos e fraudes não detectadas
   - Escalar para 50% → 100% ao validar

3. **Configuração no Go:**
   ```go
   cfg := shield.Config{
       ModelWeight: 0.7,  // 70% peso do modelo
       RulesWeight: 0.3,  // 30% peso das regras
       Threshold:   0.5,  // Threshold final combinado
   }
   ```

---

### Checklist para produção

- [ ] Treinar com dados reais (mínimo 50k transações)
- [ ] Validar com dados de um período diferente (time-based split)
- [ ] Testar latência em ambiente Go (target: < 5ms p99)
- [ ] Configurar monitoramento de drift
- [ ] Documentar decisão de threshold com stakeholders
- [ ] Implementar circuit breaker (fallback para regras se modelo falhar)
- [ ] Setup pipeline de retreinamento automatizado

In [ ]:
# --- Final summary ---
print("\n" + "=" * 60)
print("  BEANS SHIELD — RESUMO DO TREINAMENTO")
print("=" * 60)
print(f"")
print(f"  Modelo:          LightGBM (GBDT)")
print(f"  Features:        {len(FEATURE_NAMES)}")
print(f"  Amostras treino: {len(X_train):,}")
print(f"  Amostras teste:  {len(X_test):,}")
print(f"")
print(f"  CV AUC:          {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})")
print(f"  Test AUC:        {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"  Threshold ótimo: {optimal_threshold:.2f}")
print(f"  Fraudes bloqueadas: {fraud_blocked:.1%}")
print(f"  Falso positivo:    {false_positive_rate:.2%}")
print(f"")
print(f"  Artefatos:")
print(f"    - {ONNX_PATH} ({os.path.getsize(ONNX_PATH)/1024:.0f} KB)")
print(f"    - {JSON_PATH} ({os.path.getsize(JSON_PATH)/1024:.0f} KB)")
print(f"")
print("=" * 60)